In [ ]:
import requests
from datetime import datetime
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from dotenv import load_dotenv, find_dotenv 
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain.chains import RetrievalQA

from download_cards import download_model_cards
from utils import *

from retriever import Retriever
from generator import Generator

# agentic RAG
from langchain.tools import tool
from langchain.agents import Tool
from langchain_community.tools.tavily_search import TavilySearchResults

_ = load_dotenv(find_dotenv())

# Indexing 

In [ ]:
repo_url = "https://github.com/evidentlyai/evidently"
repo_name = repo_url.rstrip('/').split('/')[-1]
extract_dir = f"./{repo_name}"

download_repo = True
if download_repo:
    download_and_extract_repo(repo_url, extract_dir)

py_files = get_py_files(extract_dir)

In [ ]:
# Step 1.1: load documents
#loader = DirectoryLoader('model_cards/', glob="**/*.md", loader_cls=TextLoader)
loader = DirectoryLoader('evidently/', glob="**/*.py", loader_cls=TextLoader)
documents = loader.load()
# Step 1.2: split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

In [13]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
# Step 1.3: encode chunks into vectors and store in a vector database
vectordb = FAISS.from_documents(chunks, embeddings)


In [ ]:
vectordb.save_local("vectorstore.db")


# Retrieval

In [ ]:
# Step 2: Retrieval: retrieve the Top k chunks most relevant to the question based on semantic similarity.
retriever = vectordb.as_retriever()
# as_retriever(
    # search_type="mmr", # similarity_search, similarity_score_threshold, 
    # search_kwargs={
        # "k": 2,
        # "score_threshold": 0.5
        # })
# retriever.invoke("What is the purpose of the Evidently library?")

# Generation

In [ ]:
generator = Generator(model="gpt-4o")
template = generator.format_prompt_codegen(
    system_prompt_path="prompts/system_prompt_codegen.txt", 
    user_prompt_path="prompts/user_prompt_codegen.txt")
template

## Prompt Engineering
- TODO: DSPy prompt optimization (query parsing)

In [ ]:
prompt = ChatPromptTemplate.from_template(template)
print(prompt)

In [ ]:
# Step 3: Generation: input the original question and the retrieved chunks together into LLM to generate the final answer.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.5)

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()} 
    | prompt 
    | llm
    | StrOutputParser() 
)

query = "Please generate a new python script that detects data drift for tabular data using your context?"
result = rag_chain.invoke(query)
current_time = datetime.now()
with open(f"results/generated_{current_time}.py", "w") as file:
    file.write(result)

# Naive RAG Evaluation

In [ ]:
# Import necessary libraries
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Instantiate the models
generator_llm = ChatOpenAI(model="gpt-4o-mini")
critic_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

# Create the TestsetGenerator
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# Call the generator
testset = generator.generate_with_langchain_docs(
data_transformed, 
test_size=20, 
distributions={ 
simple: 0.5, 
reasoning: 0.25, 
multi_context: 0.25}
)

# Agentic RAG

## Define tools functions

In [ ]:
# define vector search
def vector_search(query: str):
    qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
    return qa_chain.run(query)

# define web search
web_search_tool = TavilySearchResults(k=10)
# define web search
def web_search(query: str):
    return web_search_tool.run(query)

###################################################################
# create tool call for vector search and web search
@tool
def vector_search_tool(query: str) -> str:
    """Tool for searching the vector store."""
    return vector_search(query)

@tool
def web_search_tool_func(query: str) -> str:
    """Tool for performing web search."""
    return web_search(query)

####################################################################
# define tools for the agent
tools = [
    Tool(
        name="VectorStoreSearch",
        func=vector_search_tool,
        description="Use this to search the vector store for information."
    ),
    Tool(
        name="WebSearch",
        func=web_search_tool_func,
        description="Use this to perform a web search for information."
    ),
]

In [ ]:
# define system prompt
system_prompt = """Respond to the human as helpfully and accurately as possible. You have access to the following tools: {tools}
Always try the \"VectorStoreSearch\" tool first. Only use \"WebSearch\" if the vector store does not contain the required information.
Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).
Valid "action" values: "Final Answer" or {tool_names}
Provide only ONE action per $JSON_BLOB, as shown:"
```
{{
  "action": $TOOL_NAME,
  "action_input": $INPUT
}}
```
Follow this format:
Question: input question to answer
Thought: consider previous and subsequent steps
Action:
```
$JSON_BLOB
```
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
```
{{
  "action": "Final Answer",
  "action_input": "Final response to human"
}}
Begin! Reminder to ALWAYS respond with a valid json blob of a single action.
Respond directly if appropriate. Format is Action:```$JSON_BLOB```then Observation"""

In [ ]:
# human prompt
human_prompt = """{input}
{agent_scratchpad}
(reminder to always respond in a JSON blob)"""

In [ ]:
# create prompt template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", human_prompt),
    ]
)

In [ ]:
# tool render
from langchain.tools.render import render_text_description_and_args
prompt = prompt.partial(
    tools=render_text_description_and_args(list(tools)),
    tool_names=", ".join([t.name for t in tools]),
)

In [ ]:
# create rag chain
from langchain.schema.runnable import RunnablePassthrough
from langchain.agents.output_parsers import JSONAgentOutputParser
from langchain.agents.format_scratchpad import format_log_to_str
chain = (
    RunnablePassthrough.assign(
        agent_scratchpad=lambda x: format_log_to_str(x["intermediate_steps"]),
    )
    | prompt
    | llm
    | JSONAgentOutputParser()
)

In [ ]:
# create agent
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(
    agent=chain,
    tools=tools,
    handle_parsing_errors=True,
    verbose=True
)

In [ ]:
agent_executor.invoke({"input": "What types of drifts can I detect using the Evidently library?"})

# Agentic RAG Evaluation